# Exercise MegaDetector on Local Images

This notebook runs MegaDetector on a local image folder:

- `/Users/elhorte/Pictures/project-id-tests`

It is safe to run even when the folder is currently empty.

## Step 1 — Define paths

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
md_root = repo_root / "third-party" / "MegaDetector"
image_dir = Path("/Users/elhorte/Pictures/project-id-tests")
output_dir = repo_root / "outputs" / "megadetector"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_root}")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

## Step 2 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 3 — Build an image list file for MegaDetector batch inference

In [ ]:
image_list_file = output_dir / "image_list.txt"
image_list_file.write_text("\n".join(str(p) for p in images), encoding="utf-8")
print(f"Wrote: {image_list_file}")

## Step 4 — Run MegaDetector (batch mode)

This uses the MegaDetector batch detection script and writes a JSON results file.

> If your local MegaDetector checkout uses a different entry script, update the command in the next cell accordingly.

In [ ]:
import subprocess
import sys

results_json = output_dir / "megadetector_results.json"
batch_script = md_root / "megadetector" / "detection" / "run_detector_batch.py"

if not md_root.exists():
    raise FileNotFoundError(f"MegaDetector repo not found: {md_root}")
if not batch_script.exists():
    raise FileNotFoundError(f"Batch script not found: {batch_script}")
if len(images) == 0:
    raise RuntimeError("No images available for inference. Add images and rerun.")

cmd = [
    sys.executable,
    str(batch_script),
    "MDV5A",
    str(image_list_file),
    str(results_json),
    "--recursive",
]

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(f"\nDone. Results saved to: {results_json}")

## Step 5 — Preview detections summary

In [ ]:
import json
from collections import Counter

results_json = output_dir / "megadetector_results.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
images_data = data.get("images", [])

print(f"Images in results: {len(images_data)}")

detections_per_image = Counter()
for item in images_data:
    detections_per_image[len(item.get("detections", []))] += 1

print("Detection count distribution (detections -> number of images):")
for k in sorted(detections_per_image):
    print(f"  {k} -> {detections_per_image[k]}")

## Optional next step

Add a visualization cell to draw bounding boxes for quick QA once you have sample images in the folder.